In [9]:
import fastf1
import pandas as pd

In [82]:
session = fastf1.get_session(2024, 'Silverstone', 'Race')
session.load(telemetry=True, laps=True, weather=True)

core           INFO 	Loading data for British Grand Prix - Race [v3.6.1]
req            INFO 	No cached data found for session_info. Loading data...
_api           INFO 	Fetching session info data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for driver_info. Loading data...
_api           INFO 	Fetching driver list...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for session_status_data. Loading data...
_api           INFO 	Fetching session status data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for lap_count. Loading data...
_api           INFO 	Fetching lap count data...
req            INFO 	Data has been written to cache!
req            INFO 	No cached data found for track_status_data. Loading data...
_api           INFO 	Fetching track status data...
req            INFO 	Data has been written to cache!
req            INFO 	No ca

In [83]:

# Copia base
laps = session.laps.copy()

# Nome completo
laps['FullName'] = laps['Driver'].apply(lambda x: session.get_driver(x)['FullName'])

# Flag de pitstop
laps['PitStopBool'] = laps['PitInTime'].notna()

# PitStops acumulados por piloto
laps['PitStops'] = laps.groupby('Driver')['PitStopBool'].cumsum()

# Posição atual direta
laps['CurrentPosition'] = laps['Position']

# Informação do evento
laps['Country'] = session.event['Country']
laps['Year'] = session.event['EventDate'].year

# Resultados finais da corrida
results = session.results[['DriverNumber', 'Position', 'GridPosition', 'Status']].rename(
    columns={'Position': 'FinalPosition'}
)
laps = laps.merge(results, on='DriverNumber', how='left')

# ==== ✅ CORREÇÃO DO MERGE COM WEATHER ====

# Meio da volta (melhor ponto para casar condições climáticas)
laps['LapMidTime'] = laps['LapStartTime'] + laps['LapTime'] / 2
laps['LapMidTime'] = laps['LapMidTime'].fillna(laps['LapStartTime'])

# Preparar weather
weather = session.weather_data[['Time', 'TrackTemp', 'Rainfall']].copy()

# Garantir ordenação para merge_asof
laps = laps.sort_values('LapMidTime')
weather = weather.sort_values('Time')

# Merge temporal correto
laps = pd.merge_asof(
    laps,
    weather,
    left_on='LapMidTime',
    right_on='Time',
    direction='nearest',
    tolerance=pd.Timedelta(seconds=30)  # ajuste se quiser mais/menos sensibilidade
)

# ==== Seleção das colunas finais ====
cols = [
    'FullName', 'LapTime', 'LapNumber', 'Stint', 'PitStopBool', 'IsPersonalBest', 'Compound', 'TyreLife', 'FreshTyre', 'TrackStatus',
    'CurrentPosition', 'Country', 'Year', 'FinalPosition', 'GridPosition',
    'Status', 'TrackTemp', 'Rainfall'
]

final_df = laps[cols]

display(final_df.head())


,FullName,LapTime,LapNumber,Stint,PitStopBool,IsPersonalBest,Compound,TyreLife,FreshTyre,TrackStatus,CurrentPosition,Country,Year,FinalPosition,GridPosition,Status,TrackTemp,Rainfall
0,Pierre Gasly,NaT,1.0,1.0,False,False,MEDIUM,1.0,True,1,NaN,United Kingdom,2024,20.0,19.0,Did not start,33.0,False
1,George Russell,0 days 00:01:35.211000,1.0,1.0,False,False,MEDIUM,1.0,True,1,1.0,United Kingdom,2024,19.0,1.0,Retired,33.0,False
2,Lewis Hamilton,0 days 00:01:36.034000,1.0,1.0,False,False,MEDIUM,1.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,33.0,False
3,Max Verstappen,0 days 00:01:36.711000,1.0,1.0,False,False,MEDIUM,1.0,True,1,3.0,United Kingdom,2024,2.0,4.0,Finished,33.0,False
4,Lando Norris,0 days 00:01:37.347000,1.0,1.0,False,False,MEDIUM,1.0,True,1,4.0,United Kingdom,2024,3.0,3.0,Finished,33.0,False


In [88]:
final_df[final_df['FullName'] == 'Lewis Hamilton']

,FullName,LapTime,LapNumber,Stint,PitStopBool,IsPersonalBest,Compound,TyreLife,FreshTyre,TrackStatus,CurrentPosition,Country,Year,FinalPosition,GridPosition,Status,TrackTemp,Rainfall
2,Lewis Hamilton,0 days 00:01:36.034000,1.0,1.0,False,False,MEDIUM,1.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,33.0,False
21,Lewis Hamilton,0 days 00:01:31.420000,2.0,1.0,False,True,MEDIUM,2.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,33.0,False
40,Lewis Hamilton,0 days 00:01:31.716000,3.0,1.0,False,False,MEDIUM,3.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,24.3,False
59,Lewis Hamilton,0 days 00:01:31.988000,4.0,1.0,False,False,MEDIUM,4.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,24.0,False
78,Lewis Hamilton,0 days 00:01:31.677000,5.0,1.0,False,False,MEDIUM,5.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,24.2,False
97,Lewis Hamilton,0 days 00:01:31.746000,6.0,1.0,False,False,MEDIUM,6.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,25.4,False
116,Lewis Hamilton,0 days 00:01:31.675000,7.0,1.0,False,False,MEDIUM,7.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,25.8,False
135,Lewis Hamilton,0 days 00:01:31.760000,8.0,1.0,False,False,MEDIUM,8.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,26.1,False
154,Lewis Hamilton,0 days 00:01:31.564000,9.0,1.0,False,False,MEDIUM,9.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,26.0,False
173,Lewis Hamilton,0 days 00:01:31.515000,10.0,1.0,False,False,MEDIUM,10.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,25.6,False


In [ ]:
# #final_df['LapTime'] = final_df['LapTime'].dt.total_seconds()
# # Remove as linhas onde o tipo de pneu está como UNKNOWN
# final_df = final_df[final_df['Compound'] != 'UNKNOWN']
# # Removendo voltas que não possuem tempo.
# final_df = final_df[final_df['LapTime'].notna()]
# final_df = final_df.loc[final_df['TrackStatus'] == '1']
# final_df['PitStops'] = final_df['Stint'] - 1

# pitstops = final_df.pop('PitStops')

# final_df.insert(6, 'PitStops', pitstops)

In [106]:
final_df[(final_df['Rainfall'] == True) & (final_df['FullName'] == 'Lewis Hamilton')]

,FullName,LapTime,LapNumber,Stint,PitStopBool,IsPersonalBest,PitStops,Compound,TyreLife,FreshTyre,TrackStatus,CurrentPosition,Country,Year,FinalPosition,GridPosition,Status,TrackTemp,Rainfall
325,Lewis Hamilton,95.071,18.0,1.0,False,False,0.0,MEDIUM,18.0,True,1,1.0,United Kingdom,2024,1.0,2.0,Finished,23.9,True
343,Lewis Hamilton,103.711,19.0,1.0,False,False,0.0,MEDIUM,19.0,True,1,1.0,United Kingdom,2024,1.0,2.0,Finished,22.9,True
363,Lewis Hamilton,100.021,20.0,1.0,False,False,0.0,MEDIUM,20.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,22.5,True
383,Lewis Hamilton,92.156,21.0,1.0,False,False,0.0,MEDIUM,21.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,22.1,True
401,Lewis Hamilton,91.095,22.0,1.0,False,True,0.0,MEDIUM,22.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,22.0,True
419,Lewis Hamilton,91.107,23.0,1.0,False,False,0.0,MEDIUM,23.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,22.0,True
438,Lewis Hamilton,91.635,24.0,1.0,False,False,0.0,MEDIUM,24.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,22.0,True
456,Lewis Hamilton,94.988,25.0,1.0,False,False,0.0,MEDIUM,25.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,22.0,True
474,Lewis Hamilton,100.792,26.0,1.0,False,False,0.0,MEDIUM,26.0,True,1,3.0,United Kingdom,2024,1.0,2.0,Finished,21.5,True
492,Lewis Hamilton,105.183,27.0,1.0,True,False,0.0,MEDIUM,27.0,True,1,2.0,United Kingdom,2024,1.0,2.0,Finished,21.4,True


In [102]:
final_df.to_parquet("silverstone_2024.parquet", index=False, compression="gzip")
